Generate the dataset (uses `src/sample_data.py`)

**Edit `PROJECT_DIR`.**

In [ ]:
# !pip install torch

# import torch
# print(torch.cuda.is_available())

# from google.colab import drive
# drive.mount('/content/drive')

# %cd /content/drive/MyDrive/DSNSFcomp/nsf-fmrg-data-challenge

In [ ]:
%pip install tensorflow scikit-learn

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# ============================================================
# !important! make the `sample_data.py` file into folder `nsf-fmrg-data-challenge/src` of the original github repo.
# Then, put the direction of `nsf-fmrg-data-challenge` folder on your device below :)
# PROJECT_DIR = Path('/Users/jian/MyProjects/NSF-Future-challenge/nsf-fmrg-data-challenge')

PROJECT_DIR = Path.cwd().parent

print('PROJECT_DIR =', PROJECT_DIR)
# ============================================================

In [ ]:
sys.path.append(str(PROJECT_DIR / 'src'))
from sample_data import SampleGenerator, TRACK_IDS

In [ ]:
gen = SampleGenerator(PROJECT_DIR)
print('tracks:', TRACK_IDS, '| grid:', gen.frame_grid()[[0, 1, -1]], 'mm (400 frame centres)')

In [ ]:
# one sample on demand (given a track and an x)
s = gen.make_sample(8, 60.1)
print({k: (v.shape if hasattr(v, 'shape') else v) for k, v in s.items()})

## Width labels over the whole grid

`build_labels()` sweeps every frame-centre x for all tracks and returns `width_mm` + a `quality`
flag (False where x is outside a track's height coverage or an edge is clipped). 

In [ ]:
# Build geometry-based labels from actual height values and save them separately from legacy labels.
# Use stable 0.3-mm profiles, a track-calibrated threshold, and final QA.
# model_width_mm is a median-smoothed local descriptor; raw width_mm is preserved.
labels = gen.build_labels(minimum_confidence=0.70, continuity_tolerance_fraction=0.25)
out = PROJECT_DIR / "processed_data" / "labels_geometry_width.npz"
np.savez(out, **labels)
for track_id in TRACK_IDS:
    m = (labels["track_id"] == track_id) & labels["quality"]
    widths = labels["width_mm"][m] * 1000
    model_widths = labels["model_width_mm"][m] * 1000
    print(f"Track {track_id}: accepted={m.sum()}/{(labels['track_id'] == track_id).sum()}",
          f"raw_median_um={np.median(widths) if len(widths) else np.nan:.1f}",
          f"raw_std_um={np.std(widths) if len(widths) else np.nan:.1f}",
          f"model_std_um={np.std(model_widths) if len(model_widths) else np.nan:.1f}")
fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharex=True)
for ax, track_id in zip(axes.flat, TRACK_IDS):
    m = (labels["track_id"] == track_id) & labels["quality"]
    ax.plot(labels["x_mm"][m], labels["width_mm"][m] * 1000, ".", alpha=.35, label="raw boundary width")
    ax.plot(labels["x_mm"][m], labels["model_width_mm"][m] * 1000, "-", lw=1.5, label="median-smoothed target")
    ax.set(title=f"Track {track_id}", ylabel="width (µm)"); ax.grid(alpha=.3)
axes[1,0].set_xlabel("x (mm)"); axes[1,1].set_xlabel("x (mm)")
axes[0,0].legend(); fig.suptitle("Geometry-label QA: raw versus model target"); fig.tight_layout(); plt.show()
print("Saved geometry labels to:", out)


In [ ]:
## Visualization of width

In [ ]:
# Visual QA for actual-height geometry labels. Reject uncertain profiles before ML.
import pandas as pd
qa_x = [30.1, 60.1, 90.1]
fig, axes = plt.subplots(4, 3, figsize=(15, 12), sharex=True)
qa_rows = []
for row, track_id in enumerate(TRACK_IDS):
    for col, x_mm in enumerate(qa_x):
        result = gen.width_at(track_id, x_mm)
        ax = axes[row, col]
        ax.plot(result["y_mm"], result["residual_um"], lw=1, color="tab:blue")
        ax.axhline(0, color="black", lw=.8)
        if result["polarity"]:
            ax.axhline(result["polarity"] * result["threshold_um"], color="tab:orange", ls="--", lw=.8)
        ax.fill_between(result["y_mm"], result["residual_um"], 0, where=result["feature_mask"], alpha=.25, color="tab:green")
        ax.set_title(f"T{track_id:02d}, x={x_mm:.1f}: {'ACCEPT' if result['quality'] else 'REJECT'}",
                     color="tab:green" if result["quality"] else "tab:red")
        if np.isfinite(result["left_boundary_mm"]):
            ax.axvline(result["left_boundary_mm"], color="purple", ls="-.", lw=1)
            ax.axvline(result["right_boundary_mm"], color="purple", ls="-.", lw=1)
        ax.set(xlabel="cross-track y (mm)", ylabel="height residual (µm)")
        ax.grid(alpha=.3)
        qa_rows.append({"track_id":track_id, "x_mm":x_mm, "quality":result["quality"],
                        "width_um":result["width_mm"]*1000, "left_mm":result["left_boundary_mm"],
                        "right_mm":result["right_boundary_mm"], "peak_um":result["peak_height_um"],
                        "threshold_um":result["threshold_um"], "confidence":result["confidence"],
                        "rejection_reason":result["rejection_reason"]})
fig.suptitle("Height-derived width QA: profile, threshold, and segmented deposited feature", y=1.01)
fig.tight_layout(); plt.show()
display(pd.DataFrame(qa_rows))


In [ ]:
# Load validated geometry labels and leakage-safe SEM samples for all four tracks.
import pandas as pd
# Final exploratory fit: hyperparameters/epoch count were selected earlier using
# the 8/10 -> 14 experiment.  Track 21 remains completely untouched here.
SEM_TRACKS, TRAIN_TRACKS, TEST_TRACK = [8,10,14,21], [8,10,14], 21
FINAL_TRAIN_EPOCHS = 60  # Set from the separate, whole-track validation experiment.
MIN_ACCEPTED_LABELS_BY_TRACK = {8: 100, 10: 100, 14: 50, 21: 11}
label_path = PROJECT_DIR / "processed_data" / "labels_geometry_width.npz"
labels = np.load(label_path, allow_pickle=False)
required_labels = {"track_id","x_mm","width_mm","model_width_mm","quality","left_boundary_mm","right_boundary_mm","confidence"}
missing = required_labels.difference(labels.files)
if missing: raise KeyError(f"Missing geometry-label arrays: {sorted(missing)}")
for track_id in SEM_TRACKS:
    count = int(((labels["track_id"] == track_id) & labels["quality"]).sum())
    if count < MIN_ACCEPTED_LABELS_BY_TRACK[track_id]:
        raise RuntimeError(f"Track {track_id} has only {count} accepted geometry labels (minimum: {MIN_ACCEPTED_LABELS_BY_TRACK[track_id]}). Review the QA plots and calibrate the height-boundary rule before training.")
label_ids = np.array([f"T{int(t):02d}_X{float(x):05.1f}" for t,x in zip(labels["track_id"],labels["x_mm"])])
label_index = {sample_id:i for i,sample_id in enumerate(label_ids)}
sem_by_track = {}
for track_id in SEM_TRACKS:
    path = PROJECT_DIR / "Samples" / "SEM Samples" / f"SEM_samples_Track_{track_id:02d}.npz"
    with np.load(path, allow_pickle=False) as data:
        required = {"sample_ids","track_ids","x_mm","raw_slices","substrate_only_slices","substrate_masks"}
        if required.difference(data.files): raise KeyError(f"Invalid SEM file: {path}")
        reconstructed = np.where(data["substrate_masks"].astype(bool), data["raw_slices"], 0)
        if not np.array_equal(reconstructed.astype(data["substrate_only_slices"].dtype), data["substrate_only_slices"]):
            raise AssertionError(f"Safe SEM reconstruction failed for Track {track_id}")
        sem_by_track[track_id] = {"ids":data["sample_ids"].astype(str), "x":data["x_mm"].astype(float),
                                  "image":data["substrate_only_slices"].copy(), "mask":data["substrate_masks"].astype(bool).copy()}
print("Geometry-label and safe-SEM inputs loaded.")


In [ ]:
# Align geometry labels, safe SEM, and track metadata strictly by sample_id.
parts = []
for track_id in SEM_TRACKS:
    eligible = label_ids[(labels["track_id"] == track_id) & labels["quality"]]
    data = sem_by_track[track_id]; ids = np.intersect1d(data["ids"], eligible)
    if not len(ids): raise ValueError(f"No aligned accepted labels for Track {track_id}")
    lookup = {s:i for i,s in enumerate(data["ids"])}
    si = np.array([lookup[s] for s in ids]); li = np.array([label_index[s] for s in ids])
    parts.append((ids, np.full(len(ids),track_id), data["x"][si], data["image"][si], data["mask"][si], labels["model_width_mm"][li]*1000))
aligned_sample_ids = np.concatenate([p[0] for p in parts]); aligned_track_ids = np.concatenate([p[1] for p in parts])
aligned_x_mm = np.concatenate([p[2] for p in parts]); aligned_sem_images = np.concatenate([p[3] for p in parts])
aligned_sem_masks = np.concatenate([p[4] for p in parts]); aligned_width_um = np.concatenate([p[5] for p in parts]).astype(np.float32)
print(pd.DataFrame({"track":aligned_track_ids}).value_counts().sort_index())


In [ ]:
# Load the raw five-frame thermal sequence.  Do not collapse time into summary maps.
k = 2
aligned_thermal = np.stack([gen.thermal_tensor(int(t), float(x), k=k) for t,x in zip(aligned_track_ids, aligned_x_mm)])
print("Raw thermal tensors:", aligned_thermal.shape)


In [ ]:
# Final fit: Tracks 8/10/14.  Track 21 is not used for fitting or model selection.
import tensorflow as tf
from sklearn.metrics import mean_absolute_error
tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(42)
sem_input=np.stack([aligned_sem_images.astype(np.float32)/255, aligned_sem_masks.astype(np.float32)],axis=-1)
y_um=aligned_width_um
train_idx=np.flatnonzero(np.isin(aligned_track_ids,TRAIN_TRACKS)); test_idx=np.flatnonzero(aligned_track_ids==TEST_TRACK)
print("Training tracks:", np.unique(aligned_track_ids[train_idx]))
print("Untouched test track:", np.unique(aligned_track_ids[test_idx]))
thermal_scale=max(float(aligned_thermal[train_idx].max()),1.0)
# (sample, time, height, width, channel): full sequence enters the learned encoder.
thermal_sequence=(aligned_thermal.astype(np.float32) / thermal_scale)[...,None]
y_mean=float(y_um[train_idx].mean()); y_std=float(y_um[train_idx].std()); y_z=(y_um-y_mean)/y_std
x_mean=float(aligned_x_mm[train_idx].mean()); x_std=float(aligned_x_mm[train_idx].std()); x_input=((aligned_x_mm-x_mean)/x_std).astype(np.float32)[:,None]
X_sem_train=sem_input[train_idx]; X_th_train=thermal_sequence[train_idx]; X_x_train=x_input[train_idx]
sem_layer=tf.keras.Input(sem_input.shape[1:],name="sem_input"); a=tf.keras.layers.Conv2D(8,3,activation="relu",padding="same")(sem_layer); a=tf.keras.layers.MaxPool2D(2)(a); a=tf.keras.layers.Conv2D(16,3,activation="relu",padding="same")(a); a=tf.keras.layers.GlobalAveragePooling2D()(a)
# Shared spatial CNN per frame, then a GRU over the ordered five-frame sequence.
th_layer=tf.keras.Input(thermal_sequence.shape[1:],name="thermal_sequence")
b=tf.keras.layers.TimeDistributed(tf.keras.layers.Conv2D(8,5,strides=4,activation="relu",padding="same"))(th_layer)
b=tf.keras.layers.TimeDistributed(tf.keras.layers.Conv2D(16,3,strides=2,activation="relu",padding="same"))(b)
b=tf.keras.layers.TimeDistributed(tf.keras.layers.Conv2D(24,3,strides=2,activation="relu",padding="same"))(b)
b=tf.keras.layers.TimeDistributed(tf.keras.layers.GlobalAveragePooling2D())(b)
b=tf.keras.layers.GRU(32,dropout=.10)(b)
x_layer=tf.keras.Input((1,),name="x_input"); c=tf.keras.layers.Dense(8,activation="relu")(x_layer)
z=tf.keras.layers.Concatenate()([a,b,c]); z=tf.keras.layers.Dense(32,activation="relu")(z); z=tf.keras.layers.Dropout(.2)(z); out=tf.keras.layers.Dense(1)(z)
model=tf.keras.Model([sem_layer,th_layer,x_layer],out); model.compile(tf.keras.optimizers.Adam(5e-4),loss=tf.keras.losses.Huber(delta=1.0),metrics=["mae"])
checkpoint=PROJECT_DIR/"processed_data"/"final_t8_t10_t14_sequence.keras"
history=model.fit({"sem_input":X_sem_train,"thermal_sequence":X_th_train,"x_input":X_x_train},y_z[train_idx],epochs=FINAL_TRAIN_EPOCHS,batch_size=8,verbose=2)
model.save(checkpoint)
print(f"Saved final model after {FINAL_TRAIN_EPOCHS} fixed epochs: {checkpoint}")


In [ ]:
print("Thermal sequence used by the model:", thermal_sequence.shape)
print("Train-only normalization scale:", thermal_scale)
print("Sequence mean/std:", float(thermal_sequence.mean()), float(thermal_sequence.std()))


In [ ]:
# No validation is used in the final fit: Track 21 remains untouched until final evaluation.
plt.figure(figsize=(7, 4))
plt.plot(history.history["loss"], label="train loss")
plt.xlabel("epoch"); plt.ylabel("Huber loss (standardized target)"); plt.title("Final 8/10/14 training curve")
plt.grid(alpha=.3); plt.legend(); plt.show()


## Width labels over the whole grid

`build_labels()` sweeps every frame-centre x for all tracks and returns `width_mm` + a `quality`
flag (False where x is outside a track's height coverage or an edge is clipped). 

In [ ]:
# The next cell is the single final evaluation on Track 21.
# Do not use its MAE, predictions, or error table to alter this model.


In [ ]:
# Single final evaluation on the untouched Track 21.
if len(test_idx) < 20: print(f"WARNING: only {len(test_idx)} Track-21 labels; treat this as exploratory, not a final benchmark.")
pred_test=model.predict({"sem_input":sem_input[test_idx],"thermal_sequence":thermal_sequence[test_idx],"x_input":x_input[test_idx]})[:,0]*y_std+y_mean
true_test=y_um[test_idx]; baseline_test=np.full_like(true_test,y_mean)
print("Track-21 MAE model / baseline:",mean_absolute_error(true_test,pred_test),mean_absolute_error(true_test,baseline_test))
result=pd.DataFrame({"sample_id":aligned_sample_ids[test_idx],"x_mm":aligned_x_mm[test_idx],"true_width_um":true_test,"predicted_width_um":pred_test,"error_um":pred_test-true_test}); result["abs_error_um"]=result.error_um.abs(); display(result.sort_values("abs_error_um",ascending=False).head(10))


In [ ]:
# Local geometry width versus position for every track.

fig, axes = plt.subplots(2, 2, figsize=(15, 9), sharex=True, sharey=True)

for ax, track_id in zip(axes.flat, SEM_TRACKS):
    accepted = (
        (labels["track_id"] == track_id)
        & labels["quality"]
        & np.isfinite(labels["width_mm"])
    )

    x = labels["x_mm"][accepted]
    raw_width_um = labels["width_mm"][accepted] * 1000
    model_width_um = labels["model_width_mm"][accepted] * 1000

    order = np.argsort(x)
    x = x[order]
    raw_width_um = raw_width_um[order]
    model_width_um = model_width_um[order]

    ax.scatter(
        x, raw_width_um,
        s=16, alpha=0.40, color="tab:blue",
        label="raw boundary width",
    )
    ax.plot(
        x, model_width_um,
        lw=2, color="black",
        label="median-smoothed training target",
    )

    ax.set_title(
        f"Track {track_id}: n={len(x)}, "
        f"median={np.median(model_width_um):.1f} µm"
    )
    ax.set_xlabel("aligned x (mm)")
    ax.set_ylabel("local width (µm)")
    ax.grid(alpha=0.3)

axes[0, 0].legend(loc="best")
fig.suptitle("Height-derived local width versus scan position", y=1.02, fontsize=14)
fig.tight_layout()
plt.show()

In [ ]:
# True local width versus model-predicted local width for all tracks.
pred_all_um = model.predict(
    {
        "sem_input": sem_input,
        "thermal_sequence": thermal_sequence,
        "x_input": x_input,
    },
    verbose=0,
)[:, 0] * y_std + y_mean

fig, axes = plt.subplots(2, 2, figsize=(15, 9), sharex=True, sharey=True)

for ax, track_id in zip(axes.flat, SEM_TRACKS):
    mask = aligned_track_ids == track_id
    order = np.argsort(aligned_x_mm[mask])

    x = aligned_x_mm[mask][order]

    # True target: height-derived, median-smoothed local width.
    true_width_um = aligned_width_um[mask][order]

    # Predicted width: output of the trained CNN + GRU model.
    predicted_width_um = pred_all_um[mask][order]

    ax.plot(
        x, true_width_um,
        color="black", lw=2, label="True local width (height-derived target)",
    )
    ax.plot(
        x, predicted_width_um,
        color="tab:orange", lw=2, alpha=0.9,
        label="Model-predicted local width",
    )

    ax.set_title(f"Track {track_id}")
    ax.set_xlabel("aligned x (mm)")
    ax.set_ylabel("local width (µm)")
    ax.grid(alpha=0.3)

axes[0, 0].legend(loc="best")
fig.suptitle("True versus predicted local width along each track", y=1.02, fontsize=14)
fig.tight_layout()
plt.show()

## Optional multi-descriptor geometry targets

The following cells add auditable boundary, contour, roughness, waviness, multi-target, and uncertainty outputs.

In [ ]:
# 1 — True left/right boundary targets from the height-derived label extractor.
aligned_label_idx = np.array([label_index[s] for s in aligned_sample_ids])
descriptor_frame = pd.DataFrame({
    'sample_id': aligned_sample_ids, 'track_id': aligned_track_ids, 'x_mm': aligned_x_mm,
    'width_um': aligned_width_um,
    'left_boundary_um': labels['left_boundary_mm'][aligned_label_idx] * 1000,
    'right_boundary_um': labels['right_boundary_mm'][aligned_label_idx] * 1000,
})
display(descriptor_frame.head())
print('Boundary targets are true height-derived crossings, not model predictions.')

In [ ]:
# 2 — Contour deviation target: RMS departure (µm) from a smooth deposited-feature contour.
from scipy.ndimage import gaussian_filter1d
contour_deviation_um = np.full(len(descriptor_frame), np.nan, dtype=np.float32)
for i, (track_id, x_mm) in enumerate(zip(descriptor_frame.track_id, descriptor_frame.x_mm)):
    profile = gen.width_at(int(track_id), float(x_mm))
    signal = profile['smoothed_residual_um'][profile['feature_mask']]
    if signal.size >= 8 and np.isfinite(signal).all():
        contour_deviation_um[i] = np.sqrt(np.mean((signal - gaussian_filter1d(signal, 4.0)) ** 2))
descriptor_frame['contour_deviation_um'] = contour_deviation_um
print(descriptor_frame.groupby('track_id')['contour_deviation_um'].agg(['count','median','mean','std']).round(3))

In [ ]:
# 3 — Edge roughness target: local high-frequency RMS motion of both boundaries (µm).
descriptor_frame['edge_roughness_um'] = np.nan
for track_id, group in descriptor_frame.groupby('track_id', sort=False):
    idx = group.sort_values('x_mm').index.to_numpy()
    left = descriptor_frame.loc[idx, 'left_boundary_um'].to_numpy(); right = descriptor_frame.loc[idx, 'right_boundary_um'].to_numpy()
    left_resid = left - gaussian_filter1d(left, 5.0); right_resid = right - gaussian_filter1d(right, 5.0)
    descriptor_frame.loc[idx, 'edge_roughness_um'] = np.sqrt((left_resid**2 + right_resid**2) / 2.0)
print(descriptor_frame.groupby('track_id')['edge_roughness_um'].agg(['median','mean','std']).round(3))

In [ ]:
# 4 — Waviness target: local low-frequency centreline displacement (µm).
descriptor_frame['waviness_um'] = np.nan
for track_id, group in descriptor_frame.groupby('track_id', sort=False):
    idx = group.sort_values('x_mm').index.to_numpy()
    centre_um = (descriptor_frame.loc[idx, 'left_boundary_um'].to_numpy() + descriptor_frame.loc[idx, 'right_boundary_um'].to_numpy()) / 2.0
    local_centre = gaussian_filter1d(centre_um, 3.0); long_centre = gaussian_filter1d(centre_um, 15.0)
    descriptor_frame.loc[idx, 'waviness_um'] = np.abs(local_centre - long_centre)
print(descriptor_frame.groupby('track_id')['waviness_um'].agg(['median','mean','std']).round(3))

In [ ]:
# 5 — Explicit vector of geometry descriptors for multi-output learning.
DESCRIPTOR_NAMES = ['width_um', 'left_boundary_um', 'right_boundary_um', 'contour_deviation_um', 'edge_roughness_um', 'waviness_um']
descriptor_quality = np.isfinite(descriptor_frame[DESCRIPTOR_NAMES].to_numpy()).all(axis=1)
geometry_target_um = descriptor_frame.loc[descriptor_quality, DESCRIPTOR_NAMES].to_numpy(np.float32)
geometry_target_indices = np.flatnonzero(descriptor_quality)
print('Descriptor vector order:', DESCRIPTOR_NAMES)
print('Usable multi-descriptor samples:', len(geometry_target_um), '/', len(descriptor_frame))

In [ ]:
# 6 — Fresh-model builder. Each fold computes thermal/x/target normalization from its own training tracks only.
import tensorflow as tf
descriptor_track_ids = aligned_track_ids[geometry_target_indices]
descriptor_x_mm = aligned_x_mm[geometry_target_indices].astype(np.float32)
descriptor_sem = sem_input[geometry_target_indices]
descriptor_raw_thermal = aligned_thermal[geometry_target_indices].astype(np.float32)[..., None]
descriptor_meta = descriptor_frame.loc[descriptor_quality, ['sample_id','track_id','x_mm']].reset_index(drop=True)
def gaussian_nll(y_true, y_pred):
    d = tf.shape(y_true)[1]; mean, log_var = y_pred[:, :d], tf.clip_by_value(y_pred[:, d:], -8., 5.)
    return tf.reduce_mean(0.5 * (tf.exp(-log_var) * tf.square(y_true - mean) + log_var))
def build_descriptor_model():
    sem_in = tf.keras.Input(descriptor_sem.shape[1:], name='sem_input'); a = tf.keras.layers.Conv2D(8,3,activation='relu',padding='same')(sem_in); a = tf.keras.layers.MaxPool2D(2)(a); a = tf.keras.layers.Conv2D(16,3,activation='relu',padding='same')(a); a = tf.keras.layers.GlobalAveragePooling2D()(a)
    th_in = tf.keras.Input(descriptor_raw_thermal.shape[1:], name='thermal_sequence'); b = tf.keras.layers.TimeDistributed(tf.keras.layers.Conv2D(8,5,strides=4,activation='relu',padding='same'))(th_in); b = tf.keras.layers.TimeDistributed(tf.keras.layers.Conv2D(16,3,strides=2,activation='relu',padding='same'))(b); b = tf.keras.layers.TimeDistributed(tf.keras.layers.Conv2D(24,3,strides=2,activation='relu',padding='same'))(b); b = tf.keras.layers.TimeDistributed(tf.keras.layers.GlobalAveragePooling2D())(b); b = tf.keras.layers.GRU(32,dropout=.1)(b)
    x_in = tf.keras.Input((1,), name='x_input'); c = tf.keras.layers.Dense(8,activation='relu')(x_in)
    h = tf.keras.layers.Dense(48,activation='relu')(tf.keras.layers.Concatenate()([a,b,c])); h = tf.keras.layers.Dropout(.2)(h)
    model = tf.keras.Model([sem_in,th_in,x_in], tf.keras.layers.Dense(2 * len(DESCRIPTOR_NAMES))(h))
    model.compile(tf.keras.optimizers.Adam(5e-4), loss=gaussian_nll); return model
def fit_predict_descriptor_fold(train_tracks, heldout_track, model_path=None):
    train = np.isin(descriptor_track_ids, train_tracks); heldout = descriptor_track_ids == heldout_track
    if not train.any() or not heldout.any(): raise ValueError('Empty train or held-out descriptor fold')
    thermal_scale = max(float(descriptor_raw_thermal[train].max()), 1.0); sequence = descriptor_raw_thermal / thermal_scale
    y_mean = geometry_target_um[train].mean(0); y_std = geometry_target_um[train].std(0).clip(1e-6); y_z = (geometry_target_um-y_mean)/y_std
    x_mean = float(descriptor_x_mm[train].mean()); x_std = max(float(descriptor_x_mm[train].std()), 1e-6); x_z = ((descriptor_x_mm-x_mean)/x_std)[:,None]
    tf.keras.backend.clear_session(); tf.keras.utils.set_random_seed(42); fold_model = build_descriptor_model()
    fold_model.fit({'sem_input':descriptor_sem[train], 'thermal_sequence':sequence[train], 'x_input':x_z[train]}, y_z[train], epochs=FINAL_TRAIN_EPOCHS, batch_size=8, verbose=2)
    if model_path is not None: fold_model.save(model_path)
    output = fold_model.predict({'sem_input':descriptor_sem[heldout], 'thermal_sequence':sequence[heldout], 'x_input':x_z[heldout]}, verbose=0); d=len(DESCRIPTOR_NAMES)
    table = descriptor_meta.loc[heldout].reset_index(drop=True).copy()
    for j,name in enumerate(DESCRIPTOR_NAMES): table[f'true_{name}']=geometry_target_um[heldout,j]; table[f'predicted_{name}']=output[:,j]*y_std[j]+y_mean[j]; table[f'aleatoric_std_{name}']=np.exp(np.clip(output[:,d+j],-8.,5.)/2)*y_std[j]; table[f'abs_residual_{name}']=np.abs(table[f'true_{name}']-table[f'predicted_{name}'])
    table['fold_train_tracks'] = str(list(train_tracks)); table['fold_heldout_track'] = heldout_track
    return table


In [ ]:
# 7 — Three leave-one-track-out runs. Each held-out track is unseen by its fold model.
LOTO_FOLDS = [([8,10],14), ([8,14],10), ([10,14],8)]
oof_tables = []
for fold_train_tracks, fold_heldout_track in LOTO_FOLDS:
    print(f'LOTO fold: train {fold_train_tracks} -> predict Track {fold_heldout_track}')
    oof_tables.append(fit_predict_descriptor_fold(fold_train_tracks, fold_heldout_track))
oof_descriptor_predictions = pd.concat(oof_tables, ignore_index=True)
oof_path = PROJECT_DIR / 'processed_data' / 'oof_descriptor_predictions.csv'
oof_descriptor_predictions.to_csv(oof_path, index=False)
conformal_radii_um = {}
for name in DESCRIPTOR_NAMES:
    residual = oof_descriptor_predictions[f'abs_residual_{name}'].dropna().to_numpy()
    level = min(1.0, np.ceil((residual.size + 1) * .95) / residual.size)
    conformal_radii_um[name] = float(np.quantile(residual, level, method='higher'))
print('Saved out-of-track descriptor residuals to:', oof_path)
print('Pooled 95% conformal radii (µm):', conformal_radii_um)
display(oof_descriptor_predictions.head())

In [ ]:
# 8 — Final model: train all 8/10/14, predict untouched Track 21, then apply fixed pooled radii.
final_test_predictions = fit_predict_descriptor_fold([8,10,14], 21, PROJECT_DIR / 'processed_data' / 'final_t8_t10_t14_descriptors.keras')
calibrated_test_predictions = final_test_predictions.copy()
coverage = {}
for name in DESCRIPTOR_NAMES:
    radius = conformal_radii_um[name]
    calibrated_test_predictions[f'calibrated_radius_{name}'] = radius
    calibrated_test_predictions[f'lower_{name}'] = calibrated_test_predictions[f'predicted_{name}'] - radius
    calibrated_test_predictions[f'upper_{name}'] = calibrated_test_predictions[f'predicted_{name}'] + radius
    inside = (calibrated_test_predictions[f'true_{name}'] >= calibrated_test_predictions[f'lower_{name}']) & (calibrated_test_predictions[f'true_{name}'] <= calibrated_test_predictions[f'upper_{name}'])
    coverage[name] = float(inside.mean()) if len(inside) else np.nan
final_path = PROJECT_DIR / 'processed_data' / 'track21_calibrated_descriptor_predictions.csv'
calibrated_test_predictions.to_csv(final_path, index=False)
print('Exploratory Track-21 nominal-95% interval coverage:', coverage)
print('Saved calibrated Track-21 predictions to:', final_path)
display(calibrated_test_predictions.head())

In [ ]:
# 9 — Leave-one-track-out calibration diagnostics: true versus predicted descriptors.
fig, axes = plt.subplots(2, 3, figsize=(17, 9))
colors = {8:'tab:blue', 10:'tab:orange', 14:'tab:green'}
for ax, name in zip(axes.flat, DESCRIPTOR_NAMES):
    for track_id, group in oof_descriptor_predictions.groupby('fold_heldout_track'):
        ax.scatter(group[f'true_{name}'], group[f'predicted_{name}'], s=18, alpha=.65, color=colors[int(track_id)], label=f'held-out T{int(track_id)}')
    values = np.r_[oof_descriptor_predictions[f'true_{name}'].to_numpy(), oof_descriptor_predictions[f'predicted_{name}'].to_numpy()]
    lo, hi = np.nanmin(values), np.nanmax(values); ax.plot([lo,hi],[lo,hi],'k--',lw=1)
    ax.set(title=name.replace('_',' '), xlabel='true (µm)', ylabel='out-of-track prediction (µm)'); ax.grid(alpha=.3)
axes[0,0].legend(); fig.suptitle('Leave-one-track-out descriptor predictions', y=1.02); fig.tight_layout(); plt.show()

In [ ]:
# 10 — Final Track-21 predictions with pooled conformal 95% uncertainty bands.
plot_data = calibrated_test_predictions.sort_values('x_mm')
fig, axes = plt.subplots(2, 3, figsize=(18, 9), sharex=True)
for ax, name in zip(axes.flat, DESCRIPTOR_NAMES):
    x = plot_data['x_mm']
    ax.plot(x, plot_data[f'true_{name}'], 'o-', color='black', ms=4, label='true height-derived target')
    ax.plot(x, plot_data[f'predicted_{name}'], 'o-', color='tab:orange', ms=4, label='final model mean')
    ax.fill_between(x, plot_data[f'lower_{name}'], plot_data[f'upper_{name}'], color='tab:orange', alpha=.22, label='pooled conformal 95% interval')
    ax.set(title=name.replace('_',' '), xlabel='aligned x (mm)', ylabel='µm'); ax.grid(alpha=.3)
axes[0,0].legend(); fig.suptitle('Untouched Track-21 descriptors with calibrated uncertainty', y=1.02); fig.tight_layout(); plt.show()
plt.figure(figsize=(10,4)); plt.bar([n.replace('_um','').replace('_',' ') for n in DESCRIPTOR_NAMES], [coverage[n] for n in DESCRIPTOR_NAMES], color='tab:green')
plt.axhline(.95, color='crimson', ls='--', label='nominal 95%'); plt.ylim(0,1.05); plt.ylabel('fraction inside interval'); plt.title('Track-21 empirical interval coverage (exploratory)'); plt.xticks(rotation=25, ha='right'); plt.grid(axis='y',alpha=.3); plt.legend(); plt.tight_layout(); plt.show()